In [1]:
import pickle

import matplotlib
import matplotlib.pyplot as plt
import os
import mne
import numpy as np
import pandas as pd
import torch
import copy

from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import matplotlib.pylab as pylab
import gc
import quantus
from tqdm import tqdm
from captum.attr import Saliency, GradientShap

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

/home/marco/anaconda3/envs/MA_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
CUDA extension for structured kernels (Cauchy and Vandermonde multiplication) not found. Install by going to extensions/kernels/ and running `python setup.py install`, for improved speed and memory efficiency. Note that the kernel changed for state-spaces 4.0 and must be recompiled.


In [2]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60,900)

In [4]:
def load_model(cfg, start_index=100, subject_index=2, rep=1):
    if rep ==1:
        save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/model_checkpoints/finetune"
        file_path = os.path.join(save_path, f"subject_{subject_index}", 
                           f"model_checkpoint_finetune_subject_index_{subject_index}_start_idx_{start_index}_rep_0_pen.pth") 
    else:
        save_path = f"/home/marco/Documents/GitHub/tms_eeg_decoding/data/model_checkpoints_rep_{rep}/finetune"
        file_path = os.path.join(save_path, f"subject_{subject_index}", 
                    f"model_checkpoint_finetune_subject_index_{subject_index}_start_idx_{start_index}_rep_{rep}_pen.pth") 

    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    model.eval()
    model.to(device)
    return model


In [5]:
def load_data_set(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index =  subject_index
    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names= load_eeg_data(cfg)
    return all_epochs[150:,:,:900], all_labels_raw[150:], ch_names, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep

In [6]:
def gradshap_explainer(
    model, inputs, targets, baselines=None, abs=False, normalise=False, *args, **kwargs
) -> np.array:
    """Wrapper aorund captum's GradShap implementation."""
    if (inputs.dtype) == np.ndarray:
        inputs = torch.from_numpy(inputs)
    inputs = inputs.to(kwargs.get("device", None)).float()

    if baselines is None:
        baselines = torch.zeros_like(inputs).to(kwargs.get("device", None)).float()
    else:
        baselines = torch.from_numpy(baselines).to(kwargs.get("device", None)).float()
    gc.collect()
    torch.cuda.empty_cache()

    # Set model in evaluate mode.
    model.to(kwargs.get("device", None))
    model.eval()




    #baselines = torch.zeros_like(inputs).to(kwargs.get("device", None)).float()
    gs = GradientShap(model)
    explanation = (
        gs
        .attribute(inputs=inputs, target=targets, baselines=baselines, n_samples=200)
    ).cpu().data

    gc.collect()
    torch.cuda.empty_cache()

    if normalise:
        explanation = quantus.normalise_func.normalise_by_negative(explanation)

    if isinstance(explanation, torch.Tensor):
        if explanation.requires_grad:
            return explanation.cpu().detach().numpy()
        return explanation.cpu().numpy()

    return explanation

In [7]:
def compute_explanations(subject_index, rep=1):
    input_shape_st = (60, 900)
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    batch_size = cfg.training.batch_size
    
    all_epochs, all_labels_raw, ch_names, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep = load_data_set(subject_index=subject_index)
    pred_labels= np.zeros((all_epochs.shape[0]))
    uncertainties = np.zeros((all_epochs.shape[0]))
    explanations = np.zeros_like(all_epochs)
    dic = {}
    for i in tqdm(range(all_epochs.shape[0])):
        with torch.no_grad():
            start_index = i+100
            inputs = torch.from_numpy(all_epochs[i])
            inputs = inputs.to(device).float()
            inputs = inputs.unsqueeze(0)

            model = load_model(cfg, start_index, subject_index, rep=rep)
            explanations[i: i+1] = gradshap_explainer(model, inputs , 0, **{"device": device})
            #pred_mean, log_var = model(inputs)[:, 0], model(inputs)[:, 1]
            output = model(inputs)
            pred_mean = output[:, 0]
            log_var = output[:, 1]
            var = torch.exp(log_var)
            pred_label= pred_mean.cpu().numpy()
            uncertainty= var.cpu().numpy()
            pred_labels[i] = pred_label
            uncertainties[i] = uncertainty
    
    dic['predictions'] = pred_labels
    dic['uncertainties'] = uncertainties
    dic['explanations'] = explanations
    save_dir = f"gradshap_explanations_rep_{rep}"
    os.makedirs(save_dir, exist_ok=True)
    np.save(f"{save_dir}/gradshap_data_subject_{subject_index}_rep_{rep}.npy", dic)

# Usage
#cfg = load_config()
#pred_label_original, uncertainties_original = compute_predictions_and_uncertainties(all_epochs, cfg, device, subject_index)

In [8]:
for rep in [2,3,4]:
    cfg = load_config()
    for subject_index in cfg.dataset.test_subject_indices:
        compute_explanations(subject_index, rep=rep)

Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_001_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_001_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
510 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


  0%|          | 0/360 [00:00<?, ?it/s]/tmp/ipykernel_3445478/1114802088.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights = torch.load(file_path)
/tmp/ipykernel_

Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


...

  0%|          | 0/414 [00:00<?, ?it/s]/tmp/ipykernel_3445478/1114802088.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights = torch.load(file_path)
/tmp/ipykernel_

Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_102_preprocessed_combined_py.fif ...


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_102_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available
Adding metadata with 1 columns
789 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


  0%|          | 0/639 [00:00<?, ?it/s]/tmp/ipykernel_3445478/1114802088.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights = torch.load(file_path)
/tmp/ipykernel_